In [687]:
import pandas as pd
import os

In [689]:
def check_nan_columns(df):
    print("Kiểm tra NaN từng cột:\n")
    
    for col in df.columns:
        has_nan = df[col].isna().any()
        nan_count = df[col].isna().sum()
        
        print(f"Cột: {col}")
        print(f"  - Có NaN không? {has_nan}")
        print(f"  - Số lượng NaN: {nan_count}")
        


In [690]:
# folder_path = "C:/Downloads/data_filter/check"


# data_frames = {}
# ref_areas = {}


# # duyệt từng file trong folder
# for file in os.listdir(folder_path) :
    

#     if file.endswith(".csv") :
#         # tạo đường dẫn file
#         file_path = os.path.join(folder_path,file)


#         df = pd.read_csv(file_path)

#         data_frames[file] = df

#         ref_areas[file] = set(df["REF_AREA"].dropna().unique())
 


# min_file = min(ref_areas, key = lambda x : len(ref_areas[x]))
# min_ref_set = ref_areas[min_file]


# print("File có ít nước nhất:", min_file)
# print("Số nước:", len(min_ref_set))

    
# filtered_data = {}
# for file,df in data_frames.items() :
#     filter_df = df[df["REF_AREA"].isin(min_ref_set)].copy()
#     filtered_data[file] = filter_df


#     print(f"{file} : {len(filter_df["REF_AREA"].unique())}")

In [691]:
# for file,df in filtered_data.items() :
#     save_path = os.path.join(folder_path,f"filter_{file}")
#     df.to_csv(save_path,index = False)

In [ ]:
gov = pd.read_csv(r"C:/Downloads/data_filter/filter_data/filter_government_clean.csv")
emp = pd.read_csv(r"C:/Downloads/data_filter/filter_data/filter_employment_clean.csv")
population = pd.read_csv(r"C:/Downloads/data_filter/filter_data/filter_population_age_clean.csv")
birth_rate = pd.read_csv(r"C:/Downloads/data_filter/filter_data/birth_rate.csv")
fertility = pd.read_csv(r"C:/Downloads/data_filter/filter_data/filter_fertility_clean.csv")
household = pd.read_csv(r"C:/Downloads/data_filter/filter_data/filter_household_clean.csv")
tax = pd.read_csv(r"C:/Downloads/data_filter/filter_data/labour_tax.csv")

In [693]:
birth_flat = birth_rate[["Year", "REF_AREA", "Value"]].copy()
birth_flat = birth_flat.rename(columns={"Value": "birth_rate"})

fertility_flat = fertility[["Year", "REF_AREA", "Value"]].copy()
fertility_flat = fertility_flat.rename(columns={"Value": "fertility"})

In [694]:
master_df = birth_flat.merge(
    fertility_flat,
    on=["Year", "REF_AREA"],
    how="inner"
)

master_df

,Year,REF_AREA,birth_rate,fertility
0,2000,ISL,15.463,2.08
1,2000,LUX,13.199,1.76
2,2000,DEU,9.335,1.38
3,2000,SVK,10.216,1.30
4,2000,ESP,9.825,1.22
...,...,...,...,...
543,2021,ESP,7.106,1.18
544,2021,SVK,10.360,1.63
545,2021,DEU,9.566,1.58
546,2021,LUX,10.540,1.39


In [695]:
missing_value = {}
all_countries = set(master_df["REF_AREA"].unique())


for year,group in master_df.groupby("Year") : 
    current_countries = set(group["REF_AREA"].unique())
    missing = all_countries - current_countries

    if missing :
        missing_value[year] = missing

missing_value

{2011: {'BEL'}, 2012: {'BEL'}}

In [696]:
master_df = master_df[master_df["REF_AREA"] != "BEL"]

In [ ]:
master_df.to_csv("C:/Downloads/data_filter/filter_data/birth_rate__fertility.csv",index = False)

In [697]:
master_df = master_df.merge(
    emp[["Year", "REF_AREA", "Value_emp_rate"]],
    on=["Year", "REF_AREA"],
    how="left"
)

In [698]:
master_df.rename(columns= {"Value_emp_rate" : "emp_rate"},inplace= True)
master_df

,Year,REF_AREA,birth_rate,fertility,emp_rate
0,2000,ISL,15.463,2.08,88.40
1,2000,LUX,13.199,1.76,67.47
2,2000,DEU,9.335,1.38,68.79
3,2000,SVK,10.216,1.30,63.52
4,2000,ESP,9.825,1.22,60.69
...,...,...,...,...,...
523,2021,ESP,7.106,1.18,67.48
524,2021,SVK,10.360,1.63,74.64
525,2021,DEU,9.566,1.58,79.37
526,2021,LUX,10.540,1.39,74.09


In [699]:
pop_pivot = population.pivot(
    index=["Year","REF_AREA"],
    columns="Age",
    values="% Population"
).reset_index()

In [700]:
master_df = master_df.merge(
    pop_pivot[["Year","REF_AREA","20-","20-64","65+"]],
    on =["Year", "REF_AREA"],
    how="left"
)
master_df

,Year,REF_AREA,birth_rate,fertility,emp_rate,20-,20-64,65+
0,2000,ISL,15.463,2.08,88.40,30.91,57.51,11.57
1,2000,LUX,13.199,1.76,67.47,24.50,61.42,14.08
2,2000,DEU,9.335,1.38,68.79,21.24,62.31,16.45
3,2000,SVK,10.216,1.30,63.52,27.76,60.81,11.43
4,2000,ESP,9.825,1.22,60.69,21.19,62.17,16.64
...,...,...,...,...,...,...,...,...
523,2021,ESP,7.106,1.18,67.48,19.15,61.00,19.85
524,2021,SVK,10.360,1.63,74.64,20.76,62.02,17.22
525,2021,DEU,9.566,1.58,79.37,18.49,59.45,22.06
526,2021,LUX,10.540,1.39,74.09,21.14,64.18,14.68


In [701]:
household_pivot = household.pivot(
    index = ["Year","REF_AREA"],
    columns= "Expenditure",
    values= "Value_expenditure_Share %"
)
household_pivot

Expenditure    Alcohol_Tobacco  Clothing_Footwear  Communication  Education  \
Year REF_AREA                                                                 
2000 AUT                  3.52               6.96           2.75       0.67   
     BEL                  3.89               5.21           2.29       0.41   
     CHE                  4.61               2.98           1.91       0.68   
     DEU                  3.69               6.02           2.44       0.63   
     DNK                  4.77               4.54           2.02       0.79   
...                        ...                ...            ...        ...   
2021 POL                  6.42               4.70           2.35       0.77   
     PRT                  3.35               5.52           2.44       1.53   
     SVK                  5.23               4.32           3.13       1.33   
     SVN                  4.67               5.59           2.97       1.22   
     SWE                  3.42               4.06           3.23       0.31   

Expenditure     Food  Health  Household_Goods  Housing_Utilities  \
Year REF_AREA                                                      
2000 AUT       10.32    3.73             7.32              19.37   
     BEL       12.93    5.36             6.46              21.48   
     CHE        9.04   12.40             3.96              22.75   
     DEU       10.87    3.86             7.51              23.61   
     DNK       12.05    2.55             5.50              25.97   
...              ...     ...              ...                ...   
2021 POL       18.63    6.82             5.04              20.80   
     PRT       18.05    6.01             5.14              19.24   
     SVK       19.68    2.83             6.39              29.83   
     SVN       14.38    4.31             5.58              19.25   
     SWE       12.85    3.06             6.92              25.77   

Expenditure    Other_Services  Recreation_Culture  Restaurants_Hotels  \
Year REF_AREA                                                           
2000 AUT                10.56               10.52               11.06   
     BEL                13.92                9.70                5.53   
     CHE                12.67                9.11                9.86   
     DEU                11.35               11.00                5.67   
     DNK                12.67               11.24                5.48   
...                       ...                 ...                 ...   
2021 POL                11.69                5.98                3.54   
     PRT                10.47                5.10               11.22   
     SVK                 8.80                8.09                5.17   
     SVN                10.55                8.03                6.29   
     SWE                11.17               11.44                5.78   

Expenditure    Transport  
Year REF_AREA             
2000 AUT           13.22  
     BEL           12.81  
     CHE           10.02  
     DEU           13.35  
     DNK           12.43  
...                  ...  
2021 POL           13.25  
     PRT           11.93  
     SVK            5.22  
     SVN           17.15  
     SWE           11.98  

[550 rows x 12 columns]

In [702]:
columns = ["Year","REF_AREA"] + household_pivot.columns.tolist()
columns

['Year',
 'REF_AREA',
 'Alcohol_Tobacco',
 'Clothing_Footwear',
 'Communication',
 'Education',
 'Food',
 'Health',
 'Household_Goods',
 'Housing_Utilities',
 'Other_Services',
 'Recreation_Culture',
 'Restaurants_Hotels',
 'Transport']

In [703]:
master_df = master_df.merge(
    household_pivot,
    on = ["Year","REF_AREA"],
    how = "left"
)
master_df

,Year,REF_AREA,birth_rate,fertility,emp_rate,20-,20-64,65+,Alcohol_Tobacco,Clothing_Footwear,Communication,Education,Food,Health,Household_Goods,Housing_Utilities,Other_Services,Recreation_Culture,Restaurants_Hotels,Transport
0,2000,ISL,15.463,2.08,88.40,30.91,57.51,11.57,5.13,5.89,2.06,1.06,14.63,2.91,6.56,17.07,8.04,12.44,8.43,15.79
1,2000,LUX,13.199,1.76,67.47,24.50,61.42,14.08,11.89,5.19,1.58,0.49,8.62,1.96,6.61,20.23,13.42,7.04,7.19,15.77
2,2000,DEU,9.335,1.38,68.79,21.24,62.31,16.45,3.69,6.02,2.44,0.63,10.87,3.86,7.51,23.61,11.35,11.00,5.67,13.35
3,2000,SVK,10.216,1.30,63.52,27.76,60.81,11.43,6.11,5.85,3.16,0.64,19.41,0.87,4.94,29.47,6.24,8.48,7.11,7.71
4,2000,ESP,9.825,1.22,60.69,21.19,62.17,16.64,4.39,6.11,2.33,1.54,14.09,3.21,5.71,15.14,9.77,8.97,16.47,12.27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
523,2021,ESP,7.106,1.18,67.48,19.15,61.00,19.85,4.37,3.54,2.65,1.43,14.16,4.39,4.88,24.33,10.29,6.58,12.00,11.37
524,2021,SVK,10.360,1.63,74.64,20.76,62.02,17.22,5.23,4.32,3.13,1.33,19.68,2.83,6.39,29.83,8.80,8.09,5.17,5.22
525,2021,DEU,9.566,1.58,79.37,18.49,59.45,22.06,3.55,3.84,2.34,0.85,11.73,5.62,7.03,25.48,13.14,9.48,3.85,13.10
526,2021,LUX,10.540,1.39,74.09,21.14,64.18,14.68,8.81,4.38,1.42,0.97,9.64,4.54,6.81,22.13,14.87,6.58,5.45,14.39


In [704]:
gov_total = gov[gov["Spending_type"] == "Total"]
gov_total

,Year,REF_AREA,Country,Program,Spending_type,Value of GDP %
1100,2000,AUT,Autriche,family_total,Total,2.946
1101,2001,AUT,Autriche,family_total,Total,2.872
1102,2002,AUT,Autriche,family_total,Total,2.917
1103,2003,AUT,Autriche,family_total,Total,3.030
1104,2004,AUT,Autriche,family_total,Total,2.986
...,...,...,...,...,...,...
1645,2017,GBR,Royaume-Uni,family_total,Total,3.218
1646,2018,GBR,Royaume-Uni,family_total,Total,3.012
1647,2019,GBR,Royaume-Uni,family_total,Total,2.435
1648,2020,GBR,Royaume-Uni,family_total,Total,2.385


In [705]:
master_df = master_df.merge(
    gov_total[["Year","REF_AREA","Value of GDP %"]],
    on = ["Year","REF_AREA"],
    how = "left"
)


In [706]:
master_df.rename(columns= {"Value of GDP %" : "gov_spending"},inplace= True)

In [707]:
tax["Household_type"].unique()

array(['Couple, 2 children', 'Single person, no children',
       'Couple, no children'], dtype=object)

In [708]:
couple = (tax["Household_type"] == "Couple, no children") 
single = (tax["Household_type"] == "Single person, no children") & (tax["INCOME_PRINCIPAL"] == 100)
perfect = (tax["INCOME_PRINCIPAL"] == 100) & (tax["INCOME_SPOUSE"] == 67) & (tax["Household_type"] == "Couple, 2 children")
tax = tax[couple | single | perfect]

In [709]:
tax_2 = tax.copy()

In [710]:
tax_2.drop(columns=["INCOME_PRINCIPAL","INCOME_SPOUSE","Measure"],inplace=True)

In [711]:
tax_pivot = tax_2.pivot(
    index = ["Year","REF_AREA"],
    columns= "Household_type",
    values= "Net_average_tax_rate_pct"
)
tax_pivot

Household_type  Couple, 2 children  Couple, no children  \
Year REF_AREA                                             
2000 AUT                     20.09                28.83   
     BEL                     34.98                41.98   
     CHE                     12.26                17.82   
     DEU                     34.26                40.34   
     DNK                     35.81                39.85   
...                            ...                  ...   
2021 POL                     10.74                23.96   
     PRT                     20.22                24.12   
     SVK                     17.08                22.74   
     SVN                     26.00                32.99   
     SWE                     18.97                22.97   

Household_type  Single person, no children  
Year REF_AREA                               
2000 AUT                             30.96  
     BEL                             43.01  
     CHE                             17.82  
     DEU                             43.20  
     DNK                             41.45  
...                                    ...  
2021 POL                             24.27  
     PRT                             26.66  
     SVK                             23.92  
     SVN                             34.45  
     SWE                             24.40  

[550 rows x 3 columns]

In [712]:
master_df = master_df.merge(
    tax_pivot,
    on = ["Year","REF_AREA"],
    how = "left"
)
master_df

,Year,REF_AREA,birth_rate,fertility,emp_rate,20-,20-64,65+,Alcohol_Tobacco,Clothing_Footwear,...,Household_Goods,Housing_Utilities,Other_Services,Recreation_Culture,Restaurants_Hotels,Transport,gov_spending,"Couple, 2 children","Couple, no children","Single person, no children"
0,2000,ISL,15.463,2.08,88.40,30.91,57.51,11.57,5.13,5.89,...,6.56,17.07,8.04,12.44,8.43,15.79,2.123,21.88,23.34,25.44
1,2000,LUX,13.199,1.76,67.47,24.50,61.42,14.08,11.89,5.19,...,6.61,20.23,13.42,7.04,7.19,15.77,2.983,12.62,22.90,28.67
2,2000,DEU,9.335,1.38,68.79,21.24,62.31,16.45,3.69,6.02,...,7.51,23.61,11.35,11.00,5.67,13.35,2.035,34.26,40.34,43.20
3,2000,SVK,10.216,1.30,63.52,27.76,60.81,11.43,6.11,5.85,...,4.94,29.47,6.24,8.48,7.11,7.71,1.967,13.29,19.36,20.16
4,2000,ESP,9.825,1.22,60.69,21.19,62.17,16.64,4.39,6.11,...,5.71,15.14,9.77,8.97,16.47,12.27,0.887,15.65,17.89,19.85
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
523,2021,ESP,7.106,1.18,67.48,19.15,61.00,19.85,4.37,3.54,...,4.88,24.33,10.29,6.58,12.00,11.37,1.482,17.52,19.57,21.41
524,2021,SVK,10.360,1.63,74.64,20.76,62.02,17.22,5.23,4.32,...,6.39,29.83,8.80,8.09,5.17,5.22,2.080,17.08,22.74,23.92
525,2021,DEU,9.566,1.58,79.37,18.49,59.45,22.06,3.55,3.84,...,7.03,25.48,13.14,9.48,3.85,13.10,2.645,29.20,35.67,37.78
526,2021,LUX,10.540,1.39,74.09,21.14,64.18,14.68,8.81,4.38,...,6.81,22.13,14.87,6.58,5.45,14.39,3.194,18.81,25.77,31.42
